# **Time of Flight (ToF)**

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, Dense, Flatten, Dropout, MaxPooling2D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from larq.layers import QuantConv2D, QuantDense
from larq.models import summary
import larq_compute_engine as lce
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.datasets import load_files
import numpy as np
import scipy.io
import joblib
import pandas as pd
from tabulate import tabulate
import zipfile
import urllib.request

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

In [ ]:
# If the folder does not exist, download dataset, otherwise load it from memory
if not os.path.exists('./time-of-flight'):
    url = "https://www.dropbox.com/s/4txj0ob6ovy9jbr/time-of-flight.zip?dl=1"
    u = urllib.request.urlopen(url)
    data = u.read()
    u.close()
    with open("./time-of-flight.zip", "wb") as f :
        f.write(data)
    with zipfile.ZipFile("./time-of-flight.zip", 'r') as zip_ref:
        zip_ref.extractall("./")
    os.remove("./time-of-flight.zip")
for root, dirs, files in os.walk('./time-of-flight/'):
    for file in files:
        if file.endswith('.DS_Store'):
            path = os.path.join(root, file)
            print("Deleting: %s" % (path))
            if os.remove(path):
                print("Unable to delete!")
            else:
                print("Deleted...")
dataset = load_files('./time-of-flight', shuffle=True, encoding='utf-8')
dataset.data = np.array([np.fromstring(d.split('\n')[1], dtype=int, sep=',').reshape(8, 8) for d in dataset.data])
height, width = dataset.data[0].shape
number_of_channels = 1

In [ ]:
X = np.array(dataset.data, dtype='float32')
y = np.array(dataset.target)
del dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1)

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, X_test, y_test, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(X_test, y_test)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        if bin_class:
            y_pred_value = (model.predict(X_test) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(model.predict(X_test), axis=1)
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes, 
            y_pred_value, 
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value)
        confusion_matrix_list.append(confusion_matrix_value)
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
def print_table(data,headers):
    table_data = list(zip(*data))
    table = tabulate(table_data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues", "Greens", "Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

In [ ]:
# Define early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=30,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

## Hyperparameters Optimization

In [ ]:
class BinaryNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape):
        self.input_shape = input_shape

    def build(self, hp):
        kwargs = dict(input_quantizer="ste_sign",
                     kernel_quantizer="ste_sign",
                     kernel_constraint="weight_clip")

        model = Sequential()
        
        # Tune the first Conv2D layer
        filters_conv1 = hp.Int('conv1_filters', min_value=8, max_value=32, step=8)
        model.add(Conv2D(filters=filters_conv1, kernel_size=(3, 3), activation='relu', 
                        input_shape=self.input_shape))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(BatchNormalization(scale=False))

        # Tune the first QuantConv2D layer
        filters_qconv1 = hp.Int('qconv1_filters', min_value=4, max_value=16, step=4)
        model.add(QuantConv2D(filters_qconv1, kernel_size=(2, 2),
                             kernel_quantizer="ste_sign",
                             kernel_constraint="weight_clip",
                             use_bias=False))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout1', min_value=0.2, max_value=0.5)))
        model.add(Flatten())

        # Tune the QuantDense layer
        units_qdense = hp.Int('qdense_units', min_value=4, max_value=16, step=4)
        model.add(QuantDense(units_qdense, use_bias=False, **kwargs))
        model.add(BatchNormalization(scale=False))
        # Change to single output with sigmoid for binary classification
        model.add(Dense(1, activation='sigmoid'))

        # Tune learning rate
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        return model

In [ ]:
# Initialize the hypermodel
input_shape = (height, width, number_of_channels)
hypermodel = BinaryNetHyperModel(input_shape=input_shape)

# Create the tuner
tuner = kt.BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=20,
    directory='bayesian_opt',
    project_name='tof_tuning'
)

# Search for the best hyperparameters
tuner.search(
    X_train,
    y_train,
    epochs=100,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

# Get the best hyperparameters
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)
best_models_list = tuner.get_best_models(num_models=3)

In [ ]:
# Show the top 3 models
print("\nTop 3 Models Performance:")
print("-" * 50)

for idx, hps in enumerate(best_hps_list, 1):
    print(f"\nModel {idx}:")
    print("Hyperparameters:")
    for param, value in hps.values.items():
        print(f"{param}: {value}")
    print(f"Validation Accuracy: {best_models_list[idx-1].evaluate(X_val, y_val, verbose=0)[1]:.3f}")
        
# Let user choose the model
while True:
    try:
        choice = int(input("\nChoose a model (1, 2, or 3): "))
        if 1 <= choice <= 3:
            break
        print("Please enter 1, 2, or 3")
    except ValueError:
        print("Please enter a valid number")

# Use the chosen model
# model = best_models_list[choice - 1]
chosen_hps = best_hps_list[choice - 1]

print(f"\nSelected Model {choice}")
print("Final hyperparameters:")
for param, value in chosen_hps.values.items():
    print(f"{param}: {value}")

## Build BNN

In [ ]:
# All quantized layers except the first will use the same options
kwargs = dict(input_quantizer="ste_sign",
            kernel_quantizer="ste_sign",
            kernel_constraint="weight_clip")

model = Sequential()
# First Conv2D layer with tuned filters
model.add(Conv2D(filters=chosen_hps.get('conv1_filters'), kernel_size=(3, 3), activation='relu', 
                 input_shape=input_shape))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization(scale=False))

# First QuantConv2D layer with tuned filters
model.add(QuantConv2D(chosen_hps.get('qconv1_filters'), kernel_size=(2, 2),
                      kernel_quantizer="ste_sign",
                      kernel_constraint="weight_clip",
                      use_bias=False))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization(scale=False))
model.add(Dropout(rate=chosen_hps.get('dropout1')))

model.add(Flatten())

# QuantDense layer with tuned units
model.add(QuantDense(chosen_hps.get('qdense_units'), use_bias=False, **kwargs))
model.add(BatchNormalization(scale=False))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
lr = chosen_hps.get('learning_rate')
epochs = 150
batch_size = 128

In [ ]:
model.compile(
            optimizer=Adam(learning_rate=lr),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
print(summary(model))

### Train

In [ ]:
history= model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=1)

### Evaluate

In [ ]:
result_all_models_FC, confusion_matrix_FC, predicitions_FC = evaluate_NN_models([model], X_test, y_test, bin_class=True)

In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC, headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
plt.plot(history.history['val_accuracy'], color='orange', label='Validation Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("ToF_model.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model)
    flatbuffer_file.write(flatbuffer_bytes)

## Explainability - SHAP Analysis

In [ ]:
import sys
sys.path.insert(0, '.')
from explainability_analysis import run_multi


os.makedirs('inputs/ToF', exist_ok=True)
np.save('inputs/ToF/ToF_test_data.npy', X_test)
np.save('inputs/ToF/ToF_test_label.npy', y_test)

sample_indices = list(range(min(3, len(X_test))))
run_multi(
    data_dir='inputs',
    model_dir='.',
    out_dir='explainability_output/ToF',
    task='ToF',
    indices=sample_indices,
    n_shap_bg=30
)

print(f"\n✓ SHAP analysis completed for ToF samples {sample_indices}")
print(f"  Outputs saved to: explainability_output/ToF/")
